# SFRP1 靶向 siRNA 计算筛选 · 端到端演示（Notebook）

对应技术路线 6 步（生成→规则→结构→脱靶/毒性→综合排序→化学修饰），与 `python predict.py` 同一代码路径；此处用一段**合成小 CDS** 做可复现演示，避免依赖外部数据与 ViennaRNA（自动用纯 Python 结构近似）。

**前置**：在工程根目录 `siRNA_pipeline/` 执行本 Notebook；环境需 `torch` 与 `pyyaml`（`pip install -r requirements.txt`）。


In [ ]:
import random
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    raise SystemExit('请在工程根目录 siRNA_pipeline/ 运行本 Notebook')
sys.path.insert(0, str(ROOT / 'src'))

# 固定种子 + 合成 CDS（240 nt，确定性生成，无需外部数据）
rng = random.Random(42)
syn_fa = ROOT / 'outputs' / 'runs' / 'notebook_syn.fa'
syn_fa.parent.mkdir(parents=True, exist_ok=True)
seq = ''.join(rng.choice('ATGC') for _ in range(240))
syn_fa.write_text('>syn\n%s\n' % seq, encoding='utf-8')
print('合成 CDS 已写入：', syn_fa)


## 1) 构建配置并整链运行（一次命令复现）


In [ ]:
from sirna_pipeline.pipeline.orchestrator import build_config, run_pipeline

cfg = build_config(ROOT / 'configs', run_name='nb_demo', fasta_override=syn_fa)
# 本机无 ViennaRNA → 结构用近似；不把近似结果当硬过滤（回归口径）
cfg['stages']['structure']['detector_kwargs'] = {'backend': 'approx'}
cfg['stages']['ranking']['require_structure_pass'] = False

summary = run_pipeline(cfg)
print('各阶段状态：', {k: v.get('ok') for k, v in summary['stages'].items()})


## 2) 查看标准结果文件（大赛提交格式）


In [ ]:
import csv, json

res_csv = ROOT / 'outputs' / 'results' / 'results.csv'
print('文件：', res_csv, '| 行数：', sum(1 for _ in open(res_csv, encoding='utf-8')) - 1)
with open(res_csv, encoding='utf-8') as fh:
    for i, r in enumerate(csv.DictReader(fh)):
        if i >= 5:
            break
        print(r['final_rank'], '|', r['candidate_id'], '| guide=', r['guide_seq'],
              '| final_score=', r['final_score'], '| dG_duplex=', r['dG_duplex_total'])


## 3) 化学修饰建议（Top 榜第一条的修饰记法）


In [ ]:
chem_csv = ROOT / 'outputs' / 'results' / 'chemmod_top.csv'
from sirna_pipeline.common import records
if chem_csv.exists():
    cc = records.read_records(chem_csv)[0]
    print('guide_mod :', cc.get('chem_guide_mod'))
    print('sense_mod :', cc.get('chem_sense_mod'))
else:
    print('chemmod_top.csv 未生成（请检查 08_chemmod 阶段日志）')


## 复现说明
- 真实数据全链：`python predict.py`（输入 `数据集/SFRP1-mRNA.txt`，见 `configs/paths.yaml`）；
- 换靶基因：`python predict.py --fasta <其他CDS.fa>`；
- 结果溯源：每阶段 `outputs/runs/<run>/<stage>/manifest.json` + 根 `outputs/results/run_*_summary.json`
  （含特征统计、权重/α/β、aux 可用性、种子、耗时）；
- 结构/脱靶/OligoFormer 的「重环境真值」开关说明见 `README.md` 与 `docs/design/`。
